In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys
import json
from pathlib import Path

os.chdir("/home/kaariaa3/mscthesis/")
sys.path.append("./src/")  # Add module directory to path

import pandas as pd
import numpy as np

from utils.plots import calculate_metrics, calculate_accuracy
from utils.tools import get_config, collect_jobs, prettify_table, format_table, aggregate_results
from utils.constants import PRED_COLS

In [3]:
cols = [
    "model",
    "use_instructions",
    "type_of_demonstrations",
    "number_of_demonstrations",
    "total_accuracy",
    "theme_accuracy",
    "theme_precision",
    "theme_recall",
    "theme_f1",
    "topic_accuracy",
    "topic_precision",
    "topic_recall",
    "topic_f1",
    "concept_accuracy",
    "concept_precision", 
    "concept_recall",
    "concept_f1",
]

def score_table(jobs, cols, measure_hallucination_detection=True, mark_gen_errors_as_hallucinations=False):

    print(f"Processing {len(jobs)} jobs.")    
    table = {col_name: [] for col_name in cols}
    num_parse_errors = 0
    num_gen_errors = 0
    num_rows = 0
    for job in jobs:
        df = pd.read_csv(job["result_csv"], sep=";")

        num_rows += len(df)

        # LLM failed to produce proper json
        mask_parse_errors = ~(df[PRED_COLS] != '"PARSE ERROR"').all(axis=1)
        num_parse_errors += list(mask_parse_errors).count(True)

        # LLM did not follow instructions, also includes parse errors
        mask_gen_errors = ~df[PRED_COLS].isin(["\"yes\"", "\"no\""]).all(axis=1)
        num_gen_errors += list(mask_gen_errors).count(True) - list(mask_parse_errors).count(True)

        if mark_gen_errors_as_hallucinations:
            # Set labels
            df.loc[mask_gen_errors, ['themeCorrect', 'topicCorrect', 'usesAdditionalConcepts']] = ['"no"', '"no"', '"yes"']
        else: 
            # Remove erroneous rows
            df = df[~mask_gen_errors]

        config = get_config(job["config_json"])

        # Append run configs
        table["model"].append(job["model"])
        table["use_instructions"].append(config["use_instructions"])
        table["type_of_demonstrations"].append(config["type_of_demonstrations"])
        table["number_of_demonstrations"].append(config["number_of_demonstrations"])

        # Append accuracy scores
        for name, score in calculate_accuracy(df).items():
            table[name].append(score)

        # Append precision, recall, f1 per label
        metrics = calculate_metrics(df)
        for name, score in calculate_metrics(df, measure_hallucination_detection=measure_hallucination_detection).items():
            table[name].append(score)

    print(f"Processed {num_rows} rows.")
    print(f"Number of rows with JSON parse errors: {num_parse_errors}.")
    print(f"Number of rows where LLM failed to adhere to response schema: {num_gen_errors}.")
    print(f"Total number of erroneous rows: {num_parse_errors + num_gen_errors}.")
    print(f"Error rate: {(num_parse_errors + num_gen_errors) / num_rows}")

    return table

In [4]:
def bold_extreme_values(s, by_model=True):
    # Bold max for mean

    if 'mean' not in s.name and 'std' not in s.name:
        return ['' for v in s]


    if not by_model:
        if "mean" in s.name:
            is_max = s == s.max()
            return ['font-weight: bold' if v else '' for v in is_max]
        if "std" in s.name:
            is_min = s == s.min()
            return ['font-weight: bold' if v else '' for v in is_min]
    
    font_array = []

    model_level = s.index.names.index('model')
    models = s.index.get_level_values(model_level)
    models = pd.Series(list(models)).unique()

    idx = pd.IndexSlice
    
    for model in models:   
        values_by_model = s.loc[idx[model]]
        
        if "mean" in s.name:
            is_max = values_by_model == values_by_model.max()
            font_array += ['font-weight: bold' if v else '' for v in is_max]
        if "std" in s.name:
            is_min = values_by_model == values_by_model.min()
            font_array += ['font-weight: bold' if v else '' for v in is_min]

    return font_array

In [5]:
# Collect raw data
basepath = "./outputs/v6"
finished_jobs = collect_jobs(basepath)
jobs_list = [job for _, job_list in finished_jobs.items() for job in job_list] # Flatten

res = pd.DataFrame(score_table(jobs_list, cols, mark_gen_errors_as_hallucinations=False))
res = prettify_table(res)

Processing 330 jobs.
Processed 175350 rows.
Number of rows with JSON parse errors: 1054.
Number of rows where LLM failed to adhere to response schema: 199.
Total number of erroneous rows: 1253.
Error rate: 0.007145708582834332


In [13]:
agg = aggregate_results(res, ["model", "number_of_demonstrations", "type_of_demonstrations", "use_instructions"], cols[4:])

# Drop count
#agg2 = agg2.loc[idx[use_models, :, :, :], (["theme_f1", "topic_f1", "concept_f1"], ["mean"])]
agg = format_table(agg)

#agg2
# Highlight max values for each column in model group
#agg2.style.apply(bold_extreme_values, axis=0)
#print(agg2.style.apply(bold_extreme_values, axis=0).to_latex())
#print(agg2.to_latex(escape=True, sparsify=True, float_format="%.3f"))

pd.options.display.max_columns = None
pd.options.display.max_rows = None

agg.loc[pd.IndexSlice[['Llama-70B'], :, :, :], (slice(None), ["std"])]

total accuracy  \
                                                                                      std   
model     number of demonstrations type of demonstrations use instructions                  
Llama-70B 0                        none                   yes                    0.000000   
          1                        negative               no                     0.064545   
                                                          yes                    0.052768   
                                   positive               no                     0.081340   
                                                          yes                    0.011620   
          6                        mixed                  no                     0.059698   
                                                          yes                    0.019204   
                                   negative               no                     0.050132   
                                                          yes                    0.016876   
                                   positive               no                     0.058551   
                                                          yes                    0.006997   

                                                                           theme accuracy  \
                                                                                      std   
model     number of demonstrations type of demonstrations use instructions                  
Llama-70B 0                        none                   yes                    0.000000   
          1                        negative               no                     0.012361   
                                                          yes                    0.023201   
                                   positive               no                     0.009272   
                                                          yes                    0.013334   
          6                        mixed                  no                     0.010826   
                                                          yes                    0.006894   
                                   negative               no                     0.013891   
                                                          yes                    0.006411   
                                   positive               no                     0.006943   
                                                          yes                    0.003434   

                                                                           theme precision  \
                                                                                       std   
model     number of demonstrations type of demonstrations use instructions                   
Llama-70B 0                        none                   yes                     0.000000   
          1                        negative               no                      0.027344   
                                                          yes                     0.035389   
                                   positive               no                      0.005618   
                                                          yes                     0.023247   
          6                        mixed                  no                      0.004709   
                                                          yes                     0.015173   
                                   negative               no                      0.004216   
                                                          yes                     0.016080   
                                   positive               no                      0.002381   
                                                          yes                     0.004259   

                                                                           theme recall  \
                                                             

In [7]:
flattened = agg.copy()

#flattened.columns = [" ".join(list(cols)) for cols in flattened.columns]
#flattened = flattened.reset_index()

#flattened
#flattened.to_csv("./data/metrics.csv", sep=";")

,model,number of demonstrations,type of demonstrations,use instructions,total accuracy mean,total accuracy std,total accuracy count,theme accuracy mean,theme accuracy std,theme accuracy count,...,concept accuracy count,concept precision mean,concept precision std,concept precision count,concept recall mean,concept recall std,concept recall count,concept f1 mean,concept f1 std,concept f1 count
0,Llama-8B,0,none,yes,0.398131,0.008767,5,0.770841,0.005050,5,...,5,0.632032,0.006507,5,0.919273,0.006995,5,0.571063,0.014284,5
1,Llama-8B,1,negative,no,0.436403,0.047197,5,0.603075,0.050409,5,...,5,0.806681,0.169873,5,0.679131,0.210885,5,0.692879,0.126975,5
2,Llama-8B,1,negative,yes,0.465918,0.029680,5,0.663670,0.060690,5,...,5,0.600654,0.037475,5,0.970803,0.017312,5,0.455479,0.130548,5
3,Llama-8B,1,positive,no,0.449689,0.068908,5,0.627136,0.038072,5,...,5,0.811988,0.174165,5,0.553712,0.155696,5,0.668051,0.051680,5
4,Llama-8B,1,positive,yes,0.535581,0.055835,5,0.766292,0.031736,5,...,5,0.697885,0.071814,5,0.911273,0.046611,5,0.668218,0.114600,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,Qwen3-8B,6,mixed,yes,0.593573,0.021760,5,0.733837,0.020682,5,...,5,1.000000,0.000000,5,0.361815,0.065421,5,0.748802,0.019797,5
62,Qwen3-8B,6,negative,no,0.429112,0.051232,5,0.561815,0.051256,5,...,5,0.513398,0.012440,5,0.990335,0.010044,5,0.048901,0.093611,5
63,Qwen3-8B,6,negative,yes,0.619282,0.034872,5,0.762193,0.019343,5,...,5,1.000000,0.000000,5,0.317472,0.057567,5,0.739350,0.016682,5
64,Qwen3-8B,6,positive,no,0.478261,0.003781,5,0.602647,0.029571,5,...,5,0.990909,0.020328,5,0.035636,0.028909,5,0.656715,0.006186,5
